## Get list controlled vocabularies and associated xml files

In [3]:
import rdflib
from rdflib.namespace import Namespace, RDF, SKOS
import pandas as pd

DCT = Namespace("http://purl.org/dc/terms/")
OWL = Namespace("http://www.w3.org/2002/07/owl#")

def ttl_cv_to_dataframe(ttl_path):
    g = rdflib.Graph()
    g.parse(ttl_path, format="turtle")

    rows = []
    for subject in g.subjects(RDF.type, OWL.NamedIndividual):
        labels = list(g.objects(subject, SKOS.prefLabel))
        pref_label_fr = next((str(label) for label in labels if label.language == "fr"), None)
        pref_label_en = next((str(label) for label in labels if label.language == "en"), None)
        exact_match = g.value(subject, SKOS.exactMatch)
        identifier = g.value(subject, DCT.identifier)

        rows.append({
            "prefLabel_fr": pref_label_fr,
            "prefLabel_en": pref_label_en,
            "exactMatch": str(exact_match) if exact_match is not None else None,
            "identifier": str(identifier) if identifier is not None else None,
        })

    return pd.DataFrame(rows)

ttl_path = r"C:\Users\remy.ben-messaoud\Documents\python_projects\technical-documentation\docs\ttl\Age.ttl"
df_age = ttl_cv_to_dataframe(ttl_path)
df_age


,prefLabel_fr,prefLabel_en,exactMatch,identifier
0,Prénatal,Prenatal,NaN,FAG1350
1,Nouveau-né (naissance à 28j),"Infant, Newborn (birth to 28 days)",http://id.nlm.nih.gov/mesh/D007231,FAG5352
2,Nourrisson (28j à 2 ans),Infant (28 days to 2 years),http://id.nlm.nih.gov/mesh/D007223,FAG8437
3,Petite enfance (2 à 5 ans),"Child, Preschool (2 to 5 years)",http://id.nlm.nih.gov/mesh/D002675,FAG9579
4,Enfance (6 à 12 ans),Child (6 to 12 years),http://id.nlm.nih.gov/mesh/D002648,FAG2806
5,Adolescence (13 à 18 ans),Adolescent (13 to 18 years),http://id.nlm.nih.gov/mesh/D000293,FAG2098
6,Adulte (19 à 24 ans),Young Adult (19 to 24 years),http://id.nlm.nih.gov/mesh/D055815,FAG4659
7,Adulte (25 à 44 ans),Adult (25 to 44 years),http://id.nlm.nih.gov/mesh/D000328,FAG1646
8,Adulte (45 à 64 ans),Middle Aged (45 to 64 years),http://id.nlm.nih.gov/mesh/D008875,FAG7390
9,Personne âgée (65 à 79 ans),Aged (65 to 79 years),http://id.nlm.nih.gov/mesh/D000368,FAG4840


## Do it for each file of the controlled vocabularies 

In [ ]:
import os
import glob

cv_source_dir = r"C:\Users\remy.ben-messaoud\Documents\python_projects\technical-documentation\docs\ttl"
cv_output_dir = r"C:\Users\remy.ben-messaoud\Documents\python_projects\xml_processing_home\impact_to_fresh_transformation\mappings\vocabularies"

# if cv_output_dir doesn't exist, create it
if not os.path.exists(cv_output_dir):
    os.makedirs(cv_output_dir, exist_ok=True)
else:
    # else erase all files in the output directory
    for file in os.listdir(cv_output_dir):
        file_path = os.path.join(cv_output_dir, file)
        if os.path.isfile(file_path):
            os.unlink(file_path)


cv_dataframes = {}
for rdf_file in glob.glob(os.path.join(cv_source_dir, "*.ttl")):
    name = os.path.splitext(os.path.basename(rdf_file))[0]
    df = ttl_cv_to_dataframe(rdf_file)
    cv_dataframes[name] = df

    output_path = os.path.join(cv_output_dir, f"{name}.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Wrote {len(cv_dataframes)} tables to {cv_output_dir}")


Wrote 39 tables to C:\Users\remy.ben-messaoud\Documents\python_projects\xml_processing_home\impact_to_fresh_transformation\mappings\vocabularies


## rename files to match the FReSH XSD 

In [12]:
'''
Mapping

"AuthorizationAgency.csv" -> "AuthorizingAgency.csv"
"FranceRegions.csv" -> "FranceRegion.csv"
"InterventionnalStudyModel.csv" -> "InterventionalStudyModel.csv"
"Language.csv -> "OriginLang.csv"
"Datatype.csv" -> "DataType.csv"
"OrganisationType.csv" -> "FundingAgentType.csv"
"GeographicalCoverage.csv" -> "Nation.csv"

'''

# rename files in cv_output_dir
import os

rename_mapping = {
    "AuthorisationAgency.csv": "AuthorizingAgency.csv",
    "FranceRegions.csv": "FranceRegion.csv",
    "InterventionnalStudyModel.csv": "InterventionalStudyModel.csv",
    "Language.csv": "OriginLang.csv",
    "Datatype.csv": "DataType.csv",
    "OrganisationType.csv": "FundingAgentType.csv", 
    "GeographicalCoverage.csv": "Nation.csv",
    "ObservationalStudy.csv": "ObservationalStudyDesign.csv"
    
}

for old_name, new_name in rename_mapping.items():
    old_path = os.path.join(cv_output_dir, old_name)
    new_path = os.path.join(cv_output_dir, new_name)
    if os.path.exists(old_path):
        os.rename(old_path, new_path)
